In this notebook I want to compute weighted phase lag index

In [ ]:
import numpy as np
import mne
from tensorpac import Pac
from mne_connectivity import seed_target_indices
from glob import glob
import matplotlib.pyplot as plt
import pandas as pd
import warnings 
# I only want to see warnings once
warnings.filterwarnings('ignore')
from mne.stats import permutation_cluster_test, permutation_cluster_1samp_test, ttest_ind_no_p, ttest_1samp_no_p


In [ ]:
import sys
sys.path.append('/Users/salmanqasim/Documents/GitRepos/LFPAnalysis')

In [ ]:
from LFPAnalysis import oscillation_utils


In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site=='UI']
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]


days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

In [ ]:
# Explicitly define a list of analysis events and the baseline event. Should correspond to the dict

evs = {'gamble_start':[-1.0, 0], 
       'feedback_start': [0, 1.5],
       'baseline_start': [0, 0.75]}

baseline_ev = 'baseline_start'
analysis_evs = ['feedback_start']

day = 'Day1'

for i in analysis_evs: 
    if i not in evs.keys(): 
        raise ValueError('This analysis event is not in the dictionary')

In [ ]:
# From Bahramsharif 2013
# get where phase frequencies start / finish / number
phase_frequency = 'theta'
f_pha_start, f_pha_end = 4, 8
f_pha_nb = 30
f_pha_mid = np.linspace(f_pha_start, f_pha_end, f_pha_nb)
f_pha = np.c_[f_pha_mid - f_pha_mid / 4., f_pha_mid + f_pha_mid / 4.]
# get where amplitude frequencies start / finish / number
amp_frequency = 'hfa'

f_amp_start, f_amp_end = 70, 200
f_amp_nb = 30
f_amp_mid = np.linspace(f_amp_start, f_amp_end, f_amp_nb)
f_amp = np.c_[f_amp_mid - f_amp_mid / 8., f_amp_mid + f_amp_mid / 8.]

In [ ]:
save_dir_data = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/pac'


In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
save_dir_plots = f'{base_dir}/work/qasims01/MemoryBanditData/EMU/plots'
save_dir_data = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/pac'
# 
n_perm = 250  # number of permutations
# define the Pac object (gaussian-copula PAC) 
p = Pac(idpac=(2, 2, 0), f_pha=f_pha, f_amp=f_amp, dcomplex='wavelet')

day = 'Day1'
event = 'feedback_start'
baseline_event = 'baseline_start'
    
all_dfs = [] 

for subj_id in subj_ids[0:1]:

    # baseline_start_mem
    loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
    save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
    epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif',  preload=True) 
    
    for phase_roi in ['HPC']:
        for amp_roi in ['OFC']:

            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')

            buf_ms = 1000
            edges = buf_ms * (epochs_reref.info['sfreq']/1000)

            # construct the seed-to-target mapping based on your rois - matters most for PSI as this is directional 
            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, phase_roi, amp_roi)
            seed_target_df.rename(columns={'seed':'phase', 'target':'amp'}, inplace=True)

            pwise_dfs = []
            # for cond in conditions: 
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['phase'][hemi],
                        seed_target_df['amp'][hemi])
                    
                    for pair in range(len(seed_to_target[0][0:1])):
                        # ph_ch_name = elec_df.loc[seed_to_target[0][pair], 'label']
                        # amp_ch_name = elec_df.loc[seed_to_target[1][pair], 'label']
                        ph_ch_name = epochs_reref.ch_names[seed_to_target[0][pair]]
                        amp_ch_name = epochs_reref.ch_names[seed_to_target[1][pair]]

                        amp_sig = np.squeeze(epochs_reref.copy().pick(amp_ch_name)._data) 
                        ph_sig = np.squeeze(epochs_reref.copy().pick(ph_ch_name)._data) 
                        
                        # print(amp_ch_name) 
                        
                        # if os.path.isfile(f'{save_dir_data}/{subj_id}_{phase_roi}-{amp_roi}_{ph_ch_name}-{amp_ch_name}_pac.npy'):
                        #     print('already ran this one')
                        #     continue

                        epoch_pac = p.filterfit(epochs_reref.info['sfreq'], 
                                         ph_sig, amp_sig,
                                         n_perm=n_perm, n_jobs=-1,
                                         edges=edges, random_state=0)

                        surr_mean = p.surrogates.mean(axis=0)
                        surr_std = p.surrogates.std(axis=0)
                        z_scored_pac = (epoch_pac-surr_mean)/surr_std

                        # Save out the z_scored pac so I can do individual analysis on it
                        np.save(f'{save_dir_data}/{subj_id}_{phase_roi}-{amp_roi}_{ph_ch_name}-{amp_ch_name}_pac', z_scored_pac)

                        # average the z-scores across phae and frequency amplitudes 
                        avg_conn = z_scored_pac.mean(axis=(0, 1))

                        # alternative: nan out non-significant pac_values, then sum
                        z_scored_pac[np.abs(z_scored_pac)<=1.96] = np.nan

                        sum_conn = np.nansum(z_scored_pac, axis=(0, 1))

                        pwise_df = pd.DataFrame(columns=['participant', 'roi', 'label', 'event', 'avg_conn'
                                                        'summ_conn'])

                        pwise_df['avg_conn'] = avg_conn
                        pwise_df['sum_conn'] = sum_conn
                        pwise_df['participant'] = subj_id
                        pwise_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
                        pwise_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]

                        pwise_df['amp_roi'] = amp_roi
                        pwise_df['phase_roi'] = phase_roi
                        pwise_df['event'] = event
                        pwise_df['band0'] = phase_frequency
                        pwise_df['band1'] = amp_frequency
                        pwise_df['trials'] = np.arange(1, len(sum_conn)+1)
                        pwise_df['label'] = f'{ph_ch_name}:{amp_ch_name}'
                        pwise_df['hemi'] = hemi


                        pwise_df = pwise_df.merge(learn_df[learn_df.participant==subj_id], on=['participant', 'trials'])

                        pwise_df['zrpe'] = (pwise_df.rpe - np.nanmean(pwise_df.rpe)) / (2*np.nanstd(pwise_df.rpe))
                        pwise_df['zpm'] = (pwise_df.DPRIME - np.nanmean(pwise_df.DPRIME)) / (2*np.nanstd(pwise_df.DPRIME))

            #             # aggregate
                        pwise_dfs.append(pwise_df)

            if len(pwise_dfs) > 0:
                all_df_pt = pd.concat(pwise_dfs)
                # to make sure I don't have to re-run subjects if i finish some:
                all_df_pt.to_csv(f'{save_dir_data}/{subj_id}_{phase_roi}-{amp_roi}_pac_{event}.csv', index=False)
                all_dfs.append(all_df_pt)

In [ ]:
f = plt.figure(figsize=(11, 5))
plt.subplot(121)
z_scored_pac = (epoch_pac-surr_mean)/surr_std
p.comodulogram(z_scored_pac, title=str(p), cmap='Spectral_r')

In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
save_dir_plots = f'{base_dir}work/qasims01/MemoryBanditData/EMU/plots'
save_dir_data = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/Connectivity/scratch/pac'
# 
n_perm = 250  # number of permutations
# define the Pac object (gaussian-copula PAC) 
p = Pac(idpac=(2, 2, 0), f_pha=f_pha, f_amp=f_amp, dcomplex='wavelet')

day = 'Day1'
event = 'feedback_start'
baseline_event = 'baseline_start'
    
all_dfs = [] 
sig_elecs = [] 

for subj_id in subj_ids[0:1]:

    # baseline_start_mem
    loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
    save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'
    epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif',  preload=True) 
    
    for phase_roi in ['HPC']:
        for amp_roi in ['OFC']:

            elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')

            buf_ms = 1000
            edges = buf_ms * (epochs_reref.info['sfreq']/1000)

            # construct the seed-to-target mapping based on your rois - matters most for PSI as this is directional 
            seed_target_df = oscillation_utils.make_seed_target_df(elec_df, epochs_reref, phase_roi, amp_roi)
            seed_target_df.rename(columns={'seed':'phase', 'target':'amp'}, inplace=True)

            pwise_dfs = []
            # for cond in conditions: 
            for hemi in ['l', 'r']:
                # first determine if ipsilateral connectivity is even possible; if not, move on. 
                # note: edit this if you want to compute contralateral connectivity 
                if hemi not in seed_target_df.index.tolist():
                    continue
                else:
                    seed_to_target = seed_target_indices(
                        seed_target_df['phase'][hemi],
                        seed_target_df['amp'][hemi])
                    
                    for pair in range(len(seed_to_target[0])):

                        ph_ch_name = epochs_reref.ch_names[seed_to_target[0][pair]]
                        amp_ch_name = epochs_reref.ch_names[seed_to_target[1][pair]]

                        z_scored_pac = np.load(f'{save_dir_data}/{subj_id}_{phase_roi}-{amp_roi}_{ph_ch_name}-{amp_ch_name}_pac.npy')

                        # t_obs, clusters, cluster_p_values, h0 = permutation_cluster_1samp_test(np.moveaxis(z_scored_pac, -1, 0), 
                        #                                 n_permutations=1000, n_jobs=-1, tail=1)
                        # if any(cluster_p_values)<=0.05:
                        #     print('sig electrode')
                        #     sig_elecs.append(subj_id)
                        #     f = plt.figure(figsize=(11, 5))
                        #     plt.subplot(121)
                        #     p.comodulogram(z_scored_pac, title=str(p), cmap='Spectral_r')

In [ ]:
hit_trials = combined_df.dropna(subset='trials_gamble').loc[(combined_df.participant==subj_id) & (combined_df.hits==True), 'trials_gamble'] - 1 
miss_trials = combined_df.dropna(subset='trials_gamble').loc[(combined_df.participant==subj_id) & (combined_df.hits==False), 'trials_gamble'] -1 
hit_trials = hit_trials.values.astype(int)
miss_trials = miss_trials.values.astype(int)

In [ ]:
# t_obs, clusters, cluster_p_values, h0 = permutation_cluster_1samp_test(np.moveaxis(z_scored_pac[:,:, miss_trials], -1, 0), 
#                                                                        stat_fun = ttest_1samp_no_p, 
#                                                                        n_permutations=1000, n_jobs=-1, tail=1)

In [ ]:
t_obs, clusters, cluster_p_values, h0 = permutation_cluster_test([np.moveaxis(z_scored_pac[:,:, hit_trials], -1, 0), 
                                                                  np.moveaxis(z_scored_pac[:,:, miss_trials], -1, 0)],
                                                                  n_permutations=1000, n_jobs=-1)

In [ ]:
cluster_p_values

In [ ]:
t_obs[np.abs(t_obs)<1.96] = np.nan
p.comodulogram(t_obs, title=str(p), 
               cmap='Spectral_r')

In [ ]:
cluster_p_values

In [ ]:
np.mean(t_obs)